# interp-engine on Colab — vLLM backend

**Arrived from the diagram's Notebook button?** The snippet you clicked is on your clipboard.
Run the install cell, then paste it into the last cell and run that.

[`interp-engine`](https://github.com/decoderesearch/interp-engine) reads activations out of a
transformer at 34 standardized points — `resid_post.10`, `mlp_act.3`, `z.7` — and an address means
the same tensor on every architecture it supports. This notebook is the runnable half of the point
diagram at **[interp-engine.org](https://interp-engine.org)**, which is the cheat sheet for where
those points sit and what other stacks call them.

Every card on the diagram has its own URL, so the one this snippet came from can be reopened or
sent to someone: `https://interp-engine.org/?arch=Qwen3ForCausalLM&point=resid_post.2` is
`resid_post` at layer 2 on Qwen 3, and `?vs=` beside it draws a second architecture to compare
against. The same table as markdown, with which backend serves what, is in
[SUPPORTED_POINTS.md](https://github.com/decoderesearch/interp-engine/blob/main/docs/SUPPORTED_POINTS.md).

## Pick a GPU runtime first

**Runtime → Change runtime type → T4 GPU**, then Save. vLLM is CUDA-only: on a CPU runtime the
install resolves and then `load_model` has no worker to start. This notebook asks for a GPU in its
metadata, so a fresh copy usually has one already — worth confirming, since a copy that has been
saved and reopened keeps whatever runtime it was last connected to.

A free T4 has 15 GB and no bf16, which fits checkpoints up to roughly 7B in fp16. Two knobs on
`load_model` are worth knowing before the first out-of-memory: `gpu_memory_utilization=0.8` leaves
vLLM less than the 90% it claims for its KV cache by default, and `max_model_len=2048` shrinks the
cache itself. Both are forwarded to vLLM verbatim.

For a backend that needs no GPU at all, switch the card to its `eager` tab and open
[the eager notebook](https://colab.research.google.com/github/decoderesearch/interp-engine/blob/main/notebooks/interp_engine_eager.ipynb)
instead.

## The snippet on your clipboard is awaited

Colab runs every cell inside an event loop, and there `run_with_cache` and its siblings raise
`NestedEventLoop` rather than nest a second one. So the Notebook button copies the **awaited** form
of the tab you pressed — `await model.capture(...)`, the same points through the same code path — and
says so in a comment at the top. Colab's kernel runs top-level `await` directly, so there is no
`asyncio.run` to add. The card's `vllm async` tab shows that form for reading on the page.

In [ ]:
# Several minutes. The vLLM wheel and its CUDA dependencies are a few GB, and pip replaces
# the torch Colab ships with the one vLLM pins -- which is why it may end by telling you to
# restart the session. Do that (Runtime -> Restart session), then carry on from the next
# cell: the packages are installed, and only this kernel needed replacing.
#
# The uninstall is what keeps torch and its companions one set. Colab builds all four for
# its own CUDA, and pip leaves alone any whose version already satisfies vLLM's pin -- so
# torch arrives from PyPI built for CUDA 13 while torchaudio stays on Colab's build for
# 12.8, and torchaudio then raises on import from inside warmup(). Removing them first
# means the set that comes back was resolved together.
!pip uninstall -q -y torch torchvision torchaudio torchcodec
!pip install -q "interp-engine[vllm]"

In [ ]:
# Gated checkpoints -- Gemma and Llama, among others -- need a Hugging Face token on an
# account that has accepted the model's terms. Keep it in Colab's Secrets (the key icon in
# the left sidebar) as HF_TOKEN with "Notebook access" on, and uncomment these two lines.
#
# Not as a literal in the cell: a notebook is the thing you share, and a pasted token is
# what leaks with it.

# from google.colab import userdata
# from huggingface_hub import login; login(userdata.get("HF_TOKEN"))

In [ ]:
# Paste the snippet here (Ctrl+V, or Cmd+V on a Mac), then run this cell.
#
# The diagram's Notebook button put it on your clipboard on the way in. If the clipboard
# turns out to be empty -- some browsers refuse the write outright -- reopen the point at
# interp-engine.org and press Copy on the card's `vllm async` tab. That tab, not `vllm`:
# a cell here runs inside an event loop, and the awaited form is the one that runs in one.
#
# The first run downloads the weights and builds vLLM's CUDA graphs, so it is minutes
# slower than every run after it in the same session.